# 6주차 Basic RAG 질의응답 실습

실제 PDF/DOCX → Document → Chunking → Embedding → Chroma → Retrieval → LLM

`_____` 부분을 채운 뒤 순서대로 실행하세요.


In [ ]:
# 1. 라이브러리 설치
!pip -q install -U langchain langchain-community langchain-text-splitters langchain-huggingface langchain-chroma chromadb sentence-transformers pypdf docx2txt transformers accelerate sentencepiece


In [ ]:
# 2. 실습 문서 업로드
from google.colab import files
uploaded = files.upload()
for filename in uploaded.keys():
    print("-", filename)


In [ ]:
# 3. PDF와 DOCX 문서 불러오기
from langchain_community.document_loaders import PyPDFLoader, Docx2txtLoader
documents = []
for filename in uploaded.keys():
    if filename.lower().endswith("_____"):
        documents.extend(PyPDFLoader(filename).load())
    elif filename.lower().endswith("_____"):
        documents.extend(Docx2txtLoader(filename).load())
print("Document 수:", len(documents))


In [ ]:
# 4. Document 내용과 Metadata 확인
for i, doc in enumerate(documents, 1):
    print(f"\n--- Document {i} ---")
    print("Metadata:", doc._____)
    print(doc._____[:700])


In [ ]:
# 5. Chunking
from langchain_text_splitters import RecursiveCharacterTextSplitter
splitter = RecursiveCharacterTextSplitter(chunk_size=_____, chunk_overlap=_____)
chunks = splitter.split_documents(_____)
print("Document 수:", len(documents))
print("Chunk 수:", len(chunks))


In [ ]:
# 6. Chunk와 출처 확인
for i, chunk in enumerate(chunks[:8], 1):
    print(f"\n--- Chunk {i} ---")
    print("출처:", chunk.metadata.get("_____"))
    print("페이지:", chunk.metadata.get("page", "-"))
    print(chunk.page_content[:500])


In [ ]:
# 7. Embedding
from langchain_huggingface import HuggingFaceEmbeddings
embedding_model = HuggingFaceEmbeddings(
    model_name="_____"
)
vector = embedding_model.embed_query("인턴십 모집인원은 몇 명인가요?")
print("Embedding 차원:", len(vector))
print(vector[:10])


In [ ]:
# 8. Chroma Vector Store
from langchain_chroma import Chroma
vectorstore = Chroma.from_documents(
    documents=_____,
    embedding=_____,
    collection_name="week6_basic_rag"
)
print("Vector Store 생성 완료")


In [ ]:
# 9. 유사도 검색
question = "제머나이소프트 인턴십 모집인원은 몇 명인가요?"
results = vectorstore.similarity_search(_____, k=_____)
for i, doc in enumerate(results, 1):
    print(f"\n[검색 결과 {i}]")
    print("출처:", doc.metadata.get("_____"))
    print(doc.page_content[:600])


In [ ]:
# 10. 여러 질문으로 Retrieval 확인
questions = [
    "제머나이소프트 인턴십에서 AI 관련 어떤 경험을 우대하나요?",
    "IPP 참가 신청서 제출기한은 언제인가요?",
    "IPP 이력서 제출기한은 언제인가요?"
]
for question in questions:
    print("\n질문:", question)
    results = vectorstore.similarity_search(_____, k=_____)
    for doc in results:
        print("출처:", doc.metadata.get("_____"))
        print(doc.page_content[:450])


In [ ]:
# 11. Retriever
retriever = vectorstore.as_retriever(search_kwargs={"k": _____})
docs = retriever.invoke("_____")
for doc in docs:
    print(doc.metadata.get("_____"))
    print(doc.page_content[:500])


In [ ]:
# 12. 오픈소스 LLM
from transformers import pipeline
generator = pipeline(
    "text2text-generation",
    model="_____",
    max_new_tokens=_____
)
print("LLM 준비 완료")


In [ ]:
# 13. Basic RAG 함수
def basic_rag(question, k=3):
    docs = vectorstore._____(question, k=k)
    context = "\n\n".join(doc._____ for doc in docs)

    prompt = f"""아래 참고 문서만 근거로 질문에 한국어로 간단하고 정확하게 답하세요.
문서에서 확인할 수 없는 내용은 추측하지 말고
'제공된 문서에서 확인할 수 없습니다.'라고 답하세요.

[참고 문서]
{_____}

[질문]
{_____}

[답변]
"""
    answer = generator(_____, max_new_tokens=160)[0]["generated_text"]
    return answer, docs


In [ ]:
# 14. RAG 질의응답
question = "제머나이소프트 인턴십 모집인원은 몇 명인가요?"
answer, docs = basic_rag(_____)
print("질문:", question)
print("답변:", answer)
print("\n검색 출처")
for doc in docs:
    print("-", doc.metadata.get("_____"), "/ page:", doc.metadata.get("page", "-"))


In [ ]:
# 15. 여러 질문 실행
questions = [
    "제머나이소프트 인턴십 모집기간은 언제인가요?",
    "제머나이소프트 인턴십 지원 시 어떤 AI 경험을 우대하나요?",
    "IPP 참가 신청서 제출기한은 언제인가요?"
]
for question in questions:
    answer, docs = basic_rag(_____)
    print("\n질문:", question)
    print("답변:", answer)


In [ ]:
# 16. 문서에 없는 질문 테스트
question = "_____"
answer, docs = basic_rag(_____)
print("질문:", question)
print("답변:", answer)
print("\n검색된 문서")
for doc in docs:
    print("-", doc.metadata.get("_____"))


In [ ]:
# 17. 원하는 질문 직접 입력
question = input("문서에 질문하세요: ")
answer, docs = basic_rag(_____)
print("\n답변:", answer)
print("\n검색 출처")
for doc in docs:
    print("-", doc.metadata.get("_____"), "/ page:", doc.metadata.get("page", "-"))
